# ROGII Wellbore Geology - ESN Inference

Load the trained Conv1D model, predict TVT for test wells.

**Runtime**: Kaggle GPU - **Author**: Samir Attrah

In [5]:
# Cell 1: Environment & Imports
import glob
import os
import sys
import pickle
import warnings

# Fix: Use KERAS_BACKEND=jax with safe pre-allocation instead of 'platform' allocator.
# XLA_PYTHON_CLIENT_ALLOCATOR='platform' releases GPU memory back to CUDA using pointers
# it no longer owns, causing CUDA_ERROR_INVALID_VALUE on Kaggle T4 GPUs.
# XLA_PYTHON_CLIENT_MEM_FRACTION=0.7 pre-allocates 70% of VRAM safely, leaving room
# for the Keras model weights without fragmenting the CUDA context.
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.7"

import jax

# Fix: Do NOT enable x64 on GPU. float64 doubles VRAM usage and causes CUDA context
# conflicts when JAX and Keras/XLA share the same device. Inference stays in float32.
# jax.config.update('jax_enable_x64', True)  # Disabled: unsafe on Kaggle GPU

import jax.numpy as jnp
import keras
import numpy as np
import polars as pl

warnings.filterwarnings("ignore")
print(f"Keras: {keras.__version__}, Backend: {keras.backend.backend()}")


Keras: 3.13.2, Backend: jax


In [6]:
# Cell 2: Configuration
import os

# Define base paths for local and Kaggle environments
if os.path.isdir("/kaggle/input"):
    # Kaggle environment
    BASE_DATA_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
    BASE_MODEL_DIR = "//kaggle/input/models/samerattrah/rogii-21-05/keras/default/49" # Example path
    OUT_DIR = "/kaggle/working"
else:
    # Local environment
    BASE_DATA_DIR = "/home/samer/Documents/competitions/ROGII/dataset"
    BASE_MODEL_DIR = "/home/samer/Documents/competitions/ROGII/outputs"
    OUT_DIR = "/home/samer/Documents/competitions/ROGII/outputs"

CONFIG = {
    "data_dir": BASE_DATA_DIR,
    "test_data_dir": os.path.join(BASE_DATA_DIR, "test"),
    # Model and scaler from DeepESN training script
    "model_path": os.path.join(BASE_MODEL_DIR, "deep_esn_model.pkl"),
    "scaler_path": os.path.join(BASE_MODEL_DIR, "deep_esn_scaler.pkl"),
    "submission_path": os.path.join(OUT_DIR, "submission.csv"),
}

# The model only requires the 6 base features for inference.
FEATURE_COLS: list[str] = [
    "MD", "X", "Y", "Z", "GR", "TVT_input"
]

BASE_FEATURE_COLS: list[str] = ["MD", "X", "Y", "Z", "GR", "TVT_input"]

os.makedirs(OUT_DIR, exist_ok=True)

print(f"Data Dir: {CONFIG['data_dir']}")
print(f"Test Data Dir: {CONFIG['test_data_dir']}")
print(f"Model Path: {CONFIG['model_path']}")


Data Dir: /kaggle/input/competitions/rogii-wellbore-geology-prediction
Test Data Dir: /kaggle/input/competitions/rogii-wellbore-geology-prediction/test
Model Path: //kaggle/input/models/samerattrah/rogii-21-05/keras/default/49/deep_esn_model.pkl


In [9]:
# Cell 3: DeepESN Class Definition
from typing import Dict, List, Optional, Tuple
from tqdm.auto import trange

class DeepESN:
    """Multi-layer Deep Echo State Network with JAX backend.

    Attributes:
        input_dim: Dimensionality of the input feature vectors.
        num_layers: Number of stacked reservoir layers.
        reservoir_size: Number of units (neurons) in each reservoir layer.
        spectral_radius: Target spectral radius for reservoir weight matrices.
        leak_rate: Leaking rate for state updates (between 0.0 and 1.0).
        input_scaling: Scaling factor applied to initial input weight matrices.
        sparsity: Fraction of zeros in reservoir connectivity matrices.
        ridge_alpha: L2 regularization strength for the linear readout solver.
    """

    def __init__(
        self,
        input_dim: int,
        num_layers: int = 3,
        reservoir_size: int = 100,
        spectral_radius: float = 0.95,
        leak_rate: float = 0.3,
        input_scaling: float = 0.5,
        sparsity: float = 0.1,
        ridge_alpha: float = 1e-3,
        seed: int = 42,
    ) -> None:
        self.input_dim = input_dim
        self.num_layers = num_layers
        self.reservoir_size = reservoir_size
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.input_scaling = input_scaling
        self.sparsity = sparsity
        self.ridge_alpha = ridge_alpha
        self.seed = seed

        self.w_in: List[jnp.ndarray] = []
        self.w_res: List[jnp.ndarray] = []
        self.w_out: Optional[jnp.ndarray] = None

        self._initialize_reservoirs()

    def _initialize_reservoirs(self) -> None:
        key = jax.random.PRNGKey(self.seed)

        for layer_idx in range(self.num_layers):
            key, key_in, key_res, key_mask = jax.random.split(key, 4)
            in_dim = self.input_dim if layer_idx == 0 else self.reservoir_size
            w_in_raw = jax.random.uniform(key_in, shape=(self.reservoir_size, in_dim), minval=-1.0, maxval=1.0)
            self.w_in.append(w_in_raw * self.input_scaling)

            w_res_raw = jax.random.uniform(key_res, shape=(self.reservoir_size, self.reservoir_size), minval=-1.0, maxval=1.0)
            mask = jax.random.uniform(key_mask, shape=(self.reservoir_size, self.reservoir_size)) < self.sparsity
            w_res_sparse = jnp.where(mask, w_res_raw, 0.0)

            eigs = np.linalg.eigvals(np.array(w_res_sparse))
            max_eig = np.max(np.abs(eigs))
            w_res_scaled = w_res_sparse * (self.spectral_radius / max_eig) if max_eig > 0 else w_res_sparse
            self.w_res.append(jnp.array(w_res_scaled))

    def _compute_reservoir_states_with_final(self, X: jnp.ndarray, initial_states: Optional[List[jnp.ndarray]] = None) -> Tuple[jnp.ndarray, List[jnp.ndarray]]:
        init_states = tuple(jnp.zeros((self.reservoir_size,), dtype=X.dtype) for _ in range(self.num_layers)) if initial_states is None else tuple(initial_states)
        dtype = X.dtype
        w_in_tuple = tuple(w.astype(dtype) for w in self.w_in)
        w_res_tuple = tuple(w.astype(dtype) for w in self.w_res)
        leak_rate = jnp.array(self.leak_rate, dtype=dtype)

        def step_fn(states, x_t):
            new_states = []
            current_in = x_t
            for l in range(self.num_layers):
                linear_comb = jnp.dot(w_in_tuple[l], current_in) + jnp.dot(w_res_tuple[l], states[l])
                h_tilde = jnp.tanh(linear_comb)
                h_l = (jnp.array(1.0, dtype=dtype) - leak_rate) * states[l] + leak_rate * h_tilde
                new_states.append(h_l)
                current_in = h_l
            combined_t = jnp.concatenate([x_t] + new_states, axis=0)
            return tuple(new_states), combined_t

        final_states, states_matrix = jax.lax.scan(step_fn, init_states, X)
        return states_matrix, list(final_states)

    def _evaluate_predictions(self, X: jnp.ndarray, chunk_size: int = 100000) -> jnp.ndarray:
        num_samples = X.shape[0]
        if num_samples <= chunk_size:
            states, _ = self._compute_reservoir_states_with_final(X)
            return jnp.dot(states, self.w_out)

        preds_list, init_states = [], None
        for c in range((num_samples + chunk_size - 1) // chunk_size):
            start_i, end_i = c * chunk_size, min((c + 1) * chunk_size, num_samples)
            chunk_states, last_states = self._compute_reservoir_states_with_final(X[start_i:end_i], initial_states=init_states)
            preds_list.append(jnp.dot(chunk_states, self.w_out))
            init_states = last_states
        return jnp.concatenate(preds_list, axis=0)

    def fit(self, X: jnp.ndarray, y: jnp.ndarray) -> Dict[str, List[float]]:
        if self.w_out is not None: raise ValueError("Model already fitted.")
        num_samples, num_features = X.shape[0], self.input_dim + self.num_layers * self.reservoir_size
        chunk_size = 100000
        StS = jnp.zeros((num_features, num_features), dtype=X.dtype)
        Sty = jnp.zeros((num_features, 1), dtype=X.dtype)
        y = y.reshape(-1, 1) if y.ndim == 1 else y

        init_states = None
        for c in trange((num_samples + chunk_size - 1) // chunk_size, desc="Training Chunks"):
            start_i, end_i = c * chunk_size, min((c + 1) * chunk_size, num_samples)
            S_chunk, last_states = self._compute_reservoir_states_with_final(X[start_i:end_i], initial_states=init_states)
            StS += S_chunk.T @ S_chunk
            Sty += S_chunk.T @ y[start_i:end_i]
            init_states = last_states

        A = StS + self.ridge_alpha * num_samples * jnp.identity(num_features, dtype=X.dtype)
        self.w_out = jnp.linalg.solve(A, Sty).flatten()
        y_pred = self._evaluate_predictions(X)
        rmse = jnp.sqrt(jnp.mean((y_pred - y.flatten()) ** 2))
        return {"loss": [float(rmse)], "rmse": [float(rmse)]}

    def predict(self, X: jnp.ndarray, chunk_size: int = 100000) -> jnp.ndarray:
        if self.w_out is None: raise ValueError("Model not fitted.")
        return self._evaluate_predictions(X, chunk_size=chunk_size)

In [ ]:
# Cell 4: Load model & scaler
import gc

# Fix: The pickled model expects the DeepESN class to be in a module named 'deep_esn'.
# To resolve the ModuleNotFoundError on loading, we can create a "fake" module in
# sys.modules. We then assign the DeepESN class (defined in Cell 3) to this fake
# module, allowing pickle to find the class definition it needs.
import sys
from types import ModuleType
deep_esn_module = ModuleType('deep_esn')
deep_esn_module.DeepESN = DeepESN  # The class from Cell 3
sys.modules['deep_esn'] = deep_esn_module

print("Loading model...")
with open(CONFIG["model_path"], "rb") as f:
    model = pickle.load(f)
print(f"DeepESN model loaded.")

print("Loading scaler...")
with open(CONFIG["scaler_path"], "rb") as f:
    scaler = pickle.load(f)
print("Scaler loaded.")

gc.collect()


Loading model...


ModuleNotFoundError: No module named 'deep_esn'

In [ ]:
# Cell 6: Prediction Logic
import gc

def preprocess(df, feature_cols):
    """Interpolates nulls in the base feature columns."""
    # Interpolate base features
    for col in BASE_FEATURE_COLS:
        if col in df.columns:
            df = df.with_columns(pl.col(col).interpolate().fill_null(strategy="forward").fill_null(strategy="backward").fill_null(0.0))
    return df


def get_submission_index(sample_sub):
    """Adds well_id and zero-based row_idx parsed from sample_submission ids."""
    # Corrected regex to match ID format: well_id_row_idx
    return sample_sub.with_columns(
        [
            pl.col("id").str.extract(r"^(.+)_(\d+)$", 1).alias("well_id"),
            pl.col("id")
            .str.extract(r"^(.+)_(\d+)$", 2)
            .cast(pl.Int64)
            .alias("row_idx"),
        ]
    )


def normalize_features(df, scaler):
    """Applies the exact scaler saved during training.

    Uses float32 throughout to avoid CUDA_ERROR_INVALID_VALUE caused by
    float64 operations conflicting with the GPU CUDA context on Kaggle T4 GPUs.
    JAX float64 on GPU requires jax_enable_x64, which conflicts with the
    Keras/XLA shared CUDA context.
    """
    feature_cols = scaler.get("feature_cols", FEATURE_COLS)
    raw_feats = df.select(feature_cols).to_numpy()
    # Fix: Use float32 instead of float64 to stay GPU-safe without jax_enable_x64.
    feats = jnp.array(raw_feats, dtype=jnp.float32)
    mean = jnp.array(scaler["feat_mean"], dtype=jnp.float32)
    std = jnp.array(scaler["feat_std"], dtype=jnp.float32)
    std = jnp.where(std == 0, 1.0, std)
    return (feats - mean) / std


def denormalize_target(yn, scaler):
    """Converts normalized model outputs back to raw TVT units (float32)."""
    target_mean = jnp.array(scaler["target_mean"], dtype=jnp.float32)
    target_std = jnp.array(scaler["target_std"], dtype=jnp.float32)
    return (jnp.array(yn, dtype=jnp.float32) * target_std) + target_mean


def smooth_prediction_zone(preds, row_idxs, window=11, anchor_weight=0.15):
    """Smooths only submitted rows and softly anchors the first row to continuity."""
    if len(row_idxs) == 0:
        return preds

    preds = preds.copy()
    zone = np.asarray(row_idxs, dtype=np.int64)
    zone_vals = preds[zone]

    if len(zone_vals) >= window:
        kernel = np.ones(window, dtype=np.float64) / window
        pad_left = window // 2
        pad_right = window - 1 - pad_left
        padded = np.pad(zone_vals, (pad_left, pad_right), mode="edge")
        zone_vals = np.convolve(padded, kernel, mode="valid")

    first_idx = int(zone[0])
    if first_idx > 0:
        zone_vals[0] = (1.0 - anchor_weight) * zone_vals[0] + anchor_weight * preds[
            first_idx - 1
        ]

    preds[zone] = zone_vals
    return preds


def predict_well(model: DeepESN, df: pl.DataFrame, scaler: dict, row_idxs=None, postprocess=True):
    """Inference for one well using the trained DeepESN model."""
    feature_cols = scaler.get("feature_cols", FEATURE_COLS)
    df = preprocess(df, feature_cols)
    feats_n = normalize_features(df, scaler)
    
    # Use the DeepESN's predict method, which handles chunking internally
    yn = model.predict(feats_n)

    # Cast back to float64 numpy only after JAX ops are done (safe host-side cast)
    yp = np.array(denormalize_target(yn, scaler), dtype=np.float64)

    if postprocess and row_idxs is not None:
        yp = smooth_prediction_zone(yp, row_idxs)
    
    return yp


In [ ]:
# Cell 6: Run Inference
sample_sub = get_submission_index(
    pl.read_csv(os.path.join(CONFIG["data_dir"], "sample_submission.csv"))
)

# Debug: check for None values in indices
agg_sub = (
    sample_sub.group_by("well_id")
    .agg(pl.min("row_idx").alias("min_idx"), pl.max("row_idx").alias("max_idx"))
)

# Filter out None values to prevent TypeError
submission_windows = {
    row["well_id"]: list(range(row["min_idx"], row["max_idx"] + 1))
    for row in agg_sub.filter(
        pl.col("min_idx").is_not_null() & pl.col("max_idx").is_not_null()
    ).iter_rows(named=True)
}
test_ids = sorted(submission_windows)

all_preds = {}
for wid in test_ids:
    print(f"Predicting {wid}...")
    well_path = os.path.join(CONFIG["test_data_dir"], f"{wid}__horizontal_well.csv")
    if not os.path.exists(well_path):
        print(f"  Warning: File not found for {wid}")
        continue
    
    try:
        df = pl.read_csv(well_path, infer_schema_length=10000)
        row_idxs = submission_windows[wid]
        all_preds[wid] = predict_well(model, df, scaler, row_idxs=row_idxs)
    except Exception as e:
        print(f"  Error predicting {wid}: {e}")

rows = []
missing = 0
for r in sample_sub.iter_rows(named=True):
    arr = all_preds.get(r["well_id"])
    if arr is None or r["row_idx"] is None or r["row_idx"] >= len(arr):
        val = 0.0
        missing += 1
    else:
        val = float(arr[r["row_idx"]])
    rows.append({"id": r["id"], "tvt": val})

if missing:
    print(f"Warning: {missing} submission rows were missing predictions and were filled with 0.0")

submission = pl.DataFrame(rows)
submission.write_csv(CONFIG["submission_path"])
print(f"Submission saved to {CONFIG['submission_path']}")
print(
    submission.select(
        pl.col("tvt").min().alias("min"),
        pl.col("tvt").max().alias("max"),
        pl.col("tvt").mean().alias("mean"),
    )
)


In [ ]:
# Cell 7: Visualise Predictions
import matplotlib.pyplot as plt

ANALYTICS_DIR = "../analytics"
os.makedirs(ANALYTICS_DIR, exist_ok=True)


def plot_predictions(all_preds, test_ids, submission_windows, max_plots=5):
    # Limit plots to avoid kernel crash during rendering
    plot_ids = test_ids[:max_plots]
    n_wells = len(plot_ids)
    if n_wells == 0:
        return

    fig, axes = plt.subplots(n_wells, 1, figsize=(15, 4 * n_wells), sharex=False)
    if n_wells == 1:
        axes = [axes]

    for i, wid in enumerate(plot_ids):
        preds = all_preds[wid]
        axes[i].plot(preds, label="Predicted TVT", color="blue", lw=1)
        rows = submission_windows.get(wid, [])
        if rows:
            axes[i].axvspan(
                rows[0], rows[-1], color="orange", alpha=0.15, label="Submission rows"
            )
        axes[i].set_title(f"Well {wid} - Predicted TVT (Conv1D)")
        axes[i].set_ylabel("TVT")
        axes[i].legend()
        axes[i].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYTICS_DIR, "test_predictions_plots_conv1d.png"))
    plt.show()


if "all_preds" in locals() and len(all_preds) > 0:
    plot_predictions(all_preds, test_ids, submission_windows)
